# Testcase: Subset of Collaborators

# Getting Started

Initially, we start by specifying the module where cells marked with the `#| export` directive will be automatically exported. 

In the following cell, `#| default_exp experiment `indicates that the exported file will be named 'experiment'. This name can be modified based on user's requirement & preferences

In [1]:
#| default_exp experiment



Next we import the necessary libraries, `FLSpec`, placement decorators (`aggregator/collaborator`)

In [ ]:
# | export

from metaflow import Flow

from openfl.experimental.interface.fl_spec import FLSpec
from openfl.experimental.placement.placement import aggregator, collaborator


class bcolors:  # NOQA: N801
    OKBLUE = "\033[94m"
    OKCYAN = "\033[96m"
    OKGREEN = "\033[92m"
    HEADER = "\033[95m"
    WARNING = "\033[93m"
    FAIL = "\033[91m"
    BOLD = "\033[1m"
    UNDERLINE = "\033[4m"
    ENDC = "\033[0m"


/home/refaix/miniforge3/envs/dir_shift/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-11-06 19:17:45,118	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Let us now define the flow of the testcase datastore cli

In [ ]:
#| export

class TestFlowSubsetCollaborators(FLSpec):
    """
    Testflow to validate working of Subset Collaborators in Federated Flow.
    """

    def __init__(self, **kwargs) -> None:
        super().__init__(**kwargs)

    @aggregator
    def start(self):
        """
        Starting the flow with random subset of collaborators
        """
        print(
            f"{bcolors.OKBLUE}Testing FederatedFlow - Starting Test for "
            + f"validating Subset of collaborators  {bcolors.ENDC}"
        )
        self.collaborators = self.runtime.collaborators

        # select subset of collaborators
        self.subset_collabrators = self.collaborators[:2]

        print(
            f"... Executing flow for {len(self.subset_collabrators)} collaborators out of Total: "
            + f"{len(self.collaborators)}"
        )

        self.next(self.test_valid_collaborators, foreach="subset_collabrators")

    @collaborator
    def test_valid_collaborators(self):
        """
        set the collaborator name
        """
        print("executing collaborator step test_valid_collaborators for "
              + f"collaborator {self.name}.")
        self.collaborator_ran = self.name
        self.next(self.join)

    @aggregator
    def join(self, inputs):
        """
        List of collaboartors ran successfully
        """
        print("inside join")
        self.collaborators_ran = [input.collaborator_ran for input in inputs]
        self.next(self.end)

    @aggregator
    def end(self):
        """
        End of the flow
        """
        print(f"End of the test case {TestFlowSubsetCollaborators.__name__} reached.")
        testcase()


def testcase():
    tc_pass_fail = {
        "passed": [], "failed": []
    }
    subset_collaborators = ["col1", "col2"]
    f = Flow("TestFlowSubsetCollaborators/")
    r = f.latest_run
    # Collaborator test_valid_collaborators step
    step = list(r)[1]
    # Aggregator join step
    join = list(r)[0]

    collaborators_ran = list(join)[0].data.collaborators_ran
    print(f"collaborators_ran: {collaborators_ran}")

    if len(list(step)) != len(subset_collaborators):
        tc_pass_fail["failed"].append(
            f"{bcolors.FAIL}...Flow only ran for {len(list(step))} "
            + f"instead of the {len(subset_collaborators)} expected "
            + f"collaborators- Testcase Failed.{bcolors.ENDC} "
        )
    else:
        tc_pass_fail["passed"].append(
            f"{bcolors.OKGREEN}Found {len(list(step))} tasks for each of the "
            + f"{len(subset_collaborators)} collaborators - "
            + f"Testcase Passed.{bcolors.ENDC}"
        )
    passed = True
    for collaborator_name in subset_collaborators:
        if collaborator_name not in collaborators_ran:
            passed = False
            tc_pass_fail["failed"].append(
                f"{bcolors.FAIL}...Flow did not execute for "
                + f"collaborator {collaborator_name}"
                + f" - Testcase Failed.{bcolors.ENDC}"
            )

    if passed:
        tc_pass_fail["passed"].append(
            f"{bcolors.OKGREEN}Flow executed for all collaborators"
            + f"- Testcase Passed.{bcolors.ENDC}"
        )
    for values in tc_pass_fail.values():
        print(*values, sep="\n")

    print(
        f"{bcolors.OKBLUE}Testing FederatedFlow - Ending test for validating "
        + f"the subset of collaborators. {bcolors.ENDC}"
    )
    if tc_pass_fail.get("failed"):
        tc_pass_fail_len = len(tc_pass_fail.get("failed"))
        raise AssertionError(
            f"{bcolors.FAIL}\n {tc_pass_fail_len} Test "
            + f"case(s) failed ... {bcolors.ENDC}"
        )


Aggregator step "start" registered
Collaborator step "aggregated_model_validation" registered
Collaborator step "train" registered
Collaborator step "local_model_validation" registered
Aggregator step "join" registered
Aggregator step "end" registered


## Workspace creation

In [ ]:
#| export

from openfl.experimental.runtime import FederatedRuntime

director_info = {
    'director_node_fqdn':'localhost',
    'director_port':50050,
    'cert_chain': None,
    'api_cert': None,
    'api_private_key': None,
}

# TODO: Is there a way to get the notebook path without passing it?
federated_runtime = FederatedRuntime(collaborators= ['env1','env2'], director=director_info, notebook_path='./testflow_subset_of_collaborators.ipynb')

In [6]:
federated_runtime.get_envoys()

['env1', 'env2']

In [ ]:
#| export

flflow = TestFlowSubsetCollaborators(checkpoint=True)
flflow.runtime = federated_runtime


In [8]:
flflow.run()


New experimental workspace directory structure:
generated_workspace
├── src
│   ├── __pycache__
│   ├── experiment.py
│   └── __init__.py
├── .workspace
├── plan
│   ├── defaults
│   ├── cols.yaml
│   ├── plan.yaml
│   └── data.yaml
└── requirements.txt

3 directories, 8 files
Aggregator step "start" registered
Collaborator step "aggregated_model_validation" registered
Collaborator step "train" registered
Collaborator step "local_model_validation" registered
Aggregator step "join" registered
Aggregator step "end" registered
Archive created at /home/refaix/openfl_dir_new/openfl/openfl-tutorials/experimental/interactive_api/testflow_datastore_cli/workspace/experiment.zip
Experiment was submitted to the director!
Aggregator step "start" registered
Collaborator step "aggregated_model_validation" registered
Collaborator step "train" registered
Collaborator step "local_model_validation" registered
Aggregator step "join" registered
Aggregator step "end" registered
Experiment ran successfully

In [9]:
vars(flflow)

{'_foreach_methods': ['aggregated_model_validation',
  'train',
  'local_model_validation',
  'join'],
 '_checkpoint': True,
 'model': Net(
   (conv1): Conv2d(1, 10, kernel_size=(5, 5), stride=(1, 1))
   (fc1): Linear(in_features=1440, out_features=10, bias=True)
 ),
 'optimizer': SGD (
 Parameter Group 0
     dampening: 0
     differentiable: False
     foreach: None
     fused: None
     lr: 0.01
     maximize: False
     momentum: 0.5
     nesterov: False
     weight_decay: 0
 ),
 'num_rounds': 3,
 'current_round': 3,
 '_runtime': FederatedRuntime,
 'private': 10,
 'execute_task_args': (<src.experiment.TestFlowDatastoreAndCli at 0x7f69b97b1be0>,
  <bound method TestFlowDatastoreAndCli.end of <src.experiment.TestFlowDatastoreAndCli object at 0x7f69b97b1be0>>,
  <bound method TestFlowDatastoreAndCli.join of <src.experiment.TestFlowDatastoreAndCli object at 0x7f69b97b1be0>>,
  {'env1': <src.experiment.TestFlowDatastoreAndCli at 0x7f69b8d51850>,
   'env2': <src.experiment.TestFlowDatast